In [0]:
# load tables

CATALOG_NAME = "workspace"
SCHEMA_NAME = "metadata"

SOURCE_TABLES = {
    "customers": f"{CATALOG_NAME}.{SCHEMA_NAME}.customers_bronze",
    "orders": f"{CATALOG_NAME}.{SCHEMA_NAME}.orders_bronze",
    "products": f"{CATALOG_NAME}.{SCHEMA_NAME}.products_bronze"
}

customers_df = spark.table(SOURCE_TABLES["customers"])
orders_df = spark.table(SOURCE_TABLES["orders"])
products_df = spark.table(SOURCE_TABLES["products"])

display(customers_df.limit(5))
display(orders_df.limit(5))
display(products_df.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, DateType, TimestampType
import re

# helper functions

def get_total_rows(df):
    return df.count()

def get_null_count(df, col_name):
    return (
        df.filter(
            F.col(col_name).isNull() |                  # nulls
            (F.col(col_name).cast("string") == "")      # blanks
        ).count())

def get_distinct_count(df, col_name):                   # can identify keys and categorical fields
    return df.select(col_name).distinct().count()

def get_sample_values(df, col_name, limit=5):           # infer meaning
    rows = (
        df.select(col_name)
        .where(F.col(col_name).isNotNull())
        .distinct()
        .limit(limit)
        .collect()
    )
    sample_values = []

    for row in rows:
        value = row[col_name]
        if value is not None:
            sample_values.append(str(value))

    return ", ".join(sample_values)

def get_min_max(df, field):                             # helps recommend ranges
    if not isinstance(field.dataType, NumericType):     # only for numerical fields
        return None, None

    stats = (
        df.select(
            F.min(F.col(field.name)).alias("min_value"),
            F.max(F.col(field.name)).alias("max_value")
        ).collect()[0])

    min_value = stats["min_value"]
    max_value = stats["max_value"]

    return str(min_value), str(max_value)

In [0]:
# profile table

def profile_table(df, table_name):
    total_rows = get_total_rows(df)
    profile_rows = []

    for field in df.schema.fields:
        col_name = field.name
        data_type = field.dataType.simpleString()

        null_count = get_null_count(df, col_name)
        if total_rows > 0:
            null_percent = round(null_count / total_rows * 100, 2)
        else:
            null_percent = 0
                
        distinct_count = get_distinct_count(df, col_name)

        sample_values = get_sample_values(df, col_name, limit=5)

        min_value, max_value = get_min_max(df, field)

        profile_row = {
            "table_name": table_name,
            "col_name": col_name,
            "data_type": data_type,
            "total_rows": total_rows,
            "null_count": null_count,
            "null_percent": null_percent,
            "distinct_count": distinct_count,
            "sample_values": sample_values,
            "min_value": min_value,
            "max_value": max_value
        }
        profile_rows.append(profile_row)
    
    return profile_rows


In [0]:
all_profile_rows = []

for table_name, table_path in SOURCE_TABLES.items():
    source_df = spark.table(table_path)

    table_profile_rows = profile_table(
        df=source_df,
        table_name=table_name
    )

    all_profile_rows.extend(table_profile_rows)

In [0]:
metadata_profile_raw_df = spark.createDataFrame(all_profile_rows)

display(metadata_profile_raw_df)

In [0]:
# infer meanings of columns

def infer_semantic_type(col_name, data_type, sample_values):
    name = col_name.lower()
    data_type = data_type.lower()
    samples = str(sample_values).lower()
    
    # categories: names, email, phone, dob, age, gender, address, location, zip, identifier, date, currency, categories, numeric, boolean, other_num, default

    if name in ["first_name", "last_name", "full_name"]:
        return "person_name"

    if "email" in name:
        return "email"
    
    if "phone" in name:
        return "phone_number"
    
    if name == "date_of_birth":
        return "date_of_birth"

    if name == "age":
        return "age"

    if name == "gender":
        return "gender"

    if "address" in name or "street" in name:
        return "street_address"

    if "zip" in name or "postal" in name:
        return "zip_code"

    if name in ["city", "state", "country"]:
        return "location"

    if name.endswith("_id") or name == "id":
        return "identifier"

    if "date" in name or "time" in name:
        return "date"

    money_keywords = ["total", "amount", "price", "cost", "tax", "discount"]
    if any(keyword in name for keyword in money_keywords):
        return "currency"

    category_keywords = ["status", "method", "category", "brand"]
    if any(keyword in name for keyword in category_keywords):
        return "category"

    if "quantity" in name or "stock" in name:
        return "numeric_measure"

    if name in ["delivered", "is_active", "active"]:
        return "boolean"

    if any(dtype in data_type for dtype in ["int", "double", "float", "decimal", "bigint"]):
        return "numeric"

    return "other"

In [0]:
def is_pii_or_sensitive(col_name, semantic_type):
    name = col_name.lower()

    # PII fields
    pii_types = [
        "person_name",
        "email",
        "phone_number",
        "street_address"
    ]
        
    if semantic_type in pii_types:
        return True
    
    # sensitive fields
    sensitive_types = [
        "date_of_birth", 
        "zip_code",
        "location",
    ] 
    
    if semantic_type in sensitive_types:
        return True

    # not PII and not sensitive
    return False


In [0]:
# assign governance tag rather than True/False

def assign_governance_tag(col_name, semantic_type):
    # PII, SENSITIVE, DEMOGRAPHIC, FINANCIAL, IDENTIFIER, NON_SENSITIVE
    name = col_name.lower()

    # Direct personally identifiable information
    if semantic_type in ["person_name", "email", "phone_number", "street_address"]:
        return "PII"

    # Personal information that may be sensitive depending on context
    if semantic_type in ["date_of_birth", "zip_code", "location"]:
        return "SENSITIVE"

    # Demographic information
    if semantic_type in ["age", "gender"]:
        return "DEMOGRAPHIC"

    # Financial or money-related fields
    if semantic_type == "currency":
        return "FINANCIAL"

    # IDs used for joins or record tracking
    if semantic_type == "identifier":
        return "IDENTIFIER"

    # Everything else
    return "NON_SENSITIVE"

In [0]:
# column descriptions

def generate_column_description(table_name, col_name, semantic_type):
    # I used LLM for descriptions
    # future versions can use live LLMs to generate descriptions

    readable = col_name.replace("_", " ")

    templates = {
        "identifier": f"Identifier used to uniquely reference or join {readable} records.",
        "email": f"Email address associated with a customer record.",
        "phone_number": f"Phone number associated with a customer record.",
        "date_of_birth": f"Customer date of birth used for demographic analysis and validation.",
        "person_name": f"Customer name field used for customer identification.",
        "street_address": f"Street address associated with a customer.",
        "zip_code": f"Postal code associated with a customer address.",
        "location": f"Location-related field describing customer geography.",
        "date": f"Date field representing {readable}.",
        "currency": f"Monetary field representing {readable}.",
        "category": f"Categorical field describing {readable}.",
        "numeric_measure": f"Numeric measurement field representing {readable}.",
        "boolean": f"Boolean indicator field representing {readable}.",
        "numeric": f"Numeric field representing {readable}.",
        "other": f"Text field representing {readable}."
    }

    return templates.get(semantic_type, f"Field representing {readable}.")

In [0]:
# enriching rows

profile_rows = [
    row.asDict()
    for row in metadata_profile_raw_df.collect()
]

enriched_rows = []

for row in profile_rows:
    col_name = row["col_name"]
    data_type = row["data_type"]
    sample_values = row["sample_values"]
    table_name = row["table_name"]

    # cell 6
    semantic_type = infer_semantic_type(col_name, data_type, sample_values)

    # cell 7
    pii_flag = is_pii_or_sensitive(col_name, semantic_type)

    # cell 8
    governance_tag = assign_governance_tag(col_name, semantic_type)

    # cell 9
    description = generate_column_description(table_name, col_name, semantic_type)

    # add enriched data back to row
    row["semantic_type"] = semantic_type
    row["is_pii_or_sensitive"] = pii_flag
    row["governance_tag"] = governance_tag
    row["description"] = description
    enriched_rows.append(row)

In [0]:
# convert enriched metadata back into a Spark DataFrame
metadata_profile_df = spark.createDataFrame(enriched_rows)

display(metadata_profile_df)

In [0]:
metadata_profile_df.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG_NAME}.{SCHEMA_NAME}.metadata_profile"
)